import importlib, sys

# 清除所有已缓存的 settings
for mod in list(sys.modules.keys()):
    if 'config' in mod:
        importlib.reload(sys.modules[mod])

from app.config import settings
print(settings.rerank_model)  # 现import importlib, sys

# 清除所有已缓存的 settings
for mod in list(sys.modules.keys()):
    if 'config' in mod:
        importlib.reload(sys.modules[mod])

from app.config import settings
print(settings.rerank_model)  # 现在应该能访问了在应该能访问了# 重排 (Reranker) 测试

测试 Reranker 的加载、打分和重排序功能。

In [21]:
import sys
sys.path.insert(0, r'D:\python\customeAgent\AICustomeRobort')

import uuid
from datetime import datetime, timezone
from app.models.document import Chunk
from app.rag.reranker import Reranker
from app.config import settings
from rich import print as rprint

## 1. 准备测试数据

构造一批模拟的 Chunk 数据，用于测试重排。

In [22]:
def make_chunk(content: str, title: str = '测试文档', idx: int = 0) -> Chunk:
    return Chunk(
        id=str(uuid.uuid4()),
        document_id=str(uuid.uuid4()),
        chunk_index=idx,
        title=title,
        content=content,
        created_at=datetime.now(timezone.utc),
    )

test_chunks = [
    make_chunk('七天无理由退换货政策：买家在签收后七天内可以申请无理由退换货，商品需保持完好状态。', '退换货政策', 1),
    make_chunk('如何修改收货地址：在订单详情页点击修改地址按钮即可更改收货信息。', '订单操作', 2),
    make_chunk('商品质量问题处理：收到商品有质量问题，请在48小时内联系客服并提供照片证据。', '售后保障', 3),
    make_chunk('支付方式支持：平台支持支付宝、微信支付、银行卡分期等多种支付方式。', '支付说明', 4),
    make_chunk('卖家发货时间：卖家需在订单确认后48小时内完成发货，逾期将自动赔付。', '发货规则', 5),
    make_chunk('积分使用规则：每笔订单可获得积分，积分可在结算时抵扣现金使用。', '积分说明', 6),
    make_chunk('优惠券发放与使用：平台会不定期发放优惠券，可在结算页面选择使用。', '活动优惠', 7),
    make_chunk('账号安全：建议设置强密码并开启双重验证，保护账号安全。', '账号安全', 8),
]

rprint(f'已创建 {len(test_chunks)} 个测试 Chunk')
for c in test_chunks:
    rprint(f'  [{c.chunk_index}] {c.title}: {c.content[:40]}...')

已创建 8 个测试 Chunk

[1] 退换货政策: 七天无理由退换货政策：买家在签收后七天内可以申请无理由退换货，商品需保持完好状态...

[2] 订单操作: 如何修改收货地址：在订单详情页点击修改地址按钮即可更改收货信息。...

[3] 售后保障: 商品质量问题处理：收到商品有质量问题，请在48小时内联系客服并提供照片证据。...

[4] 支付说明: 支付方式支持：平台支持支付宝、微信支付、银行卡分期等多种支付方式。...

[5] 发货规则: 卖家发货时间：卖家需在订单确认后48小时内完成发货，逾期将自动赔付。...

[6] 积分说明: 积分使用规则：每笔订单可获得积分，积分可在结算时抵扣现金使用。...

[7] 活动优惠: 优惠券发放与使用：平台会不定期发放优惠券，可在结算页面选择使用。...

[8] 账号安全: 账号安全：建议设置强密码并开启双重验证，保护账号安全。...

## 2. 初始化 Reranker 并检查就绪状态

In [23]:


from app.config import settings
print(settings.rerank_model)  # 现在应该能访问了rprint(f'settings.rerank_enabled = {settings.rerank_enabled}')
rprint(f'settings.rerank_enabled = {settings.rerank_model}')
rprint(f'settings.rerank_model = {getattr(settings, "rerank_model", "未配置")}')

reranker = Reranker()

rprint('正在加载重排模型...')
is_ready = reranker.ready
rprint(f'重排模型就绪: {is_ready}')

if not is_ready:
    rprint('重排模型未就绪，将使用原始顺序返回')
    rprint('检查项：')
    rprint('  1. rerank_enabled 是否为 True')
    rprint('  2. rerank_model 是否已配置')
    rprint('  3. 本地模型文件是否存在')

AttributeError: 'Settings' object has no attribute 'rerank_model'

## 3. 执行重排测试

对同一个 query 进行重排，观察结果变化。

In [4]:
import time

query = '七天无理由退换货'
top_k = 5

rprint(f'查询: {query}')
rprint(f'Top-K: {top_k}')
rprint()

rprint('重排前（原始顺序）:')
for i, c in enumerate(test_chunks[:top_k]):
    rprint(f'  {i+1}. [{c.title}] {c.content}')

rprint()

t0 = time.perf_counter()
reranked = reranker.rerank(query, test_chunks, top_k)
elapsed = (time.perf_counter() - t0) * 1000

rprint(f'重排后（耗时 {elapsed:.1f}ms）:')
for i, c in enumerate(reranked):
    rprint(f'  {i+1}. [{c.title}] {c.content}')

if reranker.ready:
    rprint()
    rprint('重排分析:')
    original_top1 = test_chunks[0].title
    reranked_top1 = reranked[0].title
    if original_top1 != reranked_top1:
        rprint(f'  Top-1 从 {original_top1} 调整为 {reranked_top1}')
    else:
        rprint(f'  Top-1 保持不变: {original_top1}')
else:
    rprint('  重排未生效，返回的是原始顺序')

查询: 七天无理由退换货

Top-K: 5

重排前（原始顺序）:

1. [退换货政策] 七天无理由退换货政策：买家在签收后七天内可以申请无理由退换货，商品需保持完好状态。

2. [订单操作] 如何修改收货地址：在订单详情页点击修改地址按钮即可更改收货信息。

...

2. [订单操作] 如何修改收货地址：在订单详情页点击修改地址按钮即可更改收货信息。

3. [售后保障] 商品质量问题处理：收到商品有质量问题，请在48小时内联系客服并提供照片证据。

4. [支付说明] 支付方式支持：平台支持支付宝、微信支付、银行卡分期等多种支付方式。

5. [发货规则] 卖家发货时间：卖家需在订单确认后48小时内完成发货，逾期将自动赔付。

重排未生效，返回的是原始顺序

## 4. 多 Query 批量测试

测试不同 query 的重排效果。

In [5]:
test_queries = [
    '七天无理由',
    '怎么改地址',
    '质量问题',
    '支付方式',
    '发货时间',
    '积分怎么用',
    '优惠券',
]

rprint('批量重排测试:')
rprint('=' * 60)

for q in test_queries:
    t0 = time.perf_counter()
    results = reranker.rerank(q, test_chunks, top_k=3)
    elapsed = (time.perf_counter() - t0) * 1000
    rprint(f'\nQuery: {q}  ({elapsed:.1f}ms)')
    for i, c in enumerate(results):
        rprint(f'  {i+1}. [{c.title}] {c.content[:50]}...')

批量重排测试:

============================================================

Query: 七天无理由  (0.0ms)

1. [退换货政策] 七天无理由退换货政策：买家在签收后七天内可以申请无理由退换货，商品需保持完好状态。...

2. [订单操作] 如何修改收货地址：在订单详情页点击修改地址按钮即可更改收货信息。...

...

3. [售后保障] 商品质量问题处理：收到商品有质量问题，请在48小时内联系客服并提供照片证据。...

Query: 优惠券  (0.0ms)

1. [退换货政策] 七天无理由退换货政策：买家在签收后七天内可以申请无理由退换货，商品需保持完好状态。...

2. [订单操作] 如何修改收货地址：在订单详情页点击修改地址按钮即可更改收货信息。...

3. [售后保障] 商品质量问题处理：收到商品有质量问题，请在48小时内联系客服并提供照片证据。...

## 5. 边界场景测试

In [6]:
rprint('边界场景测试:')
rprint()

rprint('1. 空列表测试:')
result = reranker.rerank('test', [], 5)
rprint(f'   输入 [] -> 输出 {result} (应为 [])')
assert result == [], '空列表测试失败'
rprint('   通过')

rprint('2. top_k 大于文档数:')
small_docs = test_chunks[:2]
result = reranker.rerank('七天', small_docs, 10)
rprint(f'   输入 {len(small_docs)} docs, top_k=10 -> 输出 {len(result)} docs')
assert len(result) == 2, 'top_k 溢出测试失败'
rprint('   通过')

rprint('3. top_k=1:')
result = reranker.rerank('七天', test_chunks, 1)
rprint(f'   输出 {len(result)} 条')
assert len(result) == 1, 'top_k=1 测试失败'
rprint('   通过')

rprint('4. 重复内容测试:')
dup_docs = [test_chunks[0], test_chunks[0]]
result = reranker.rerank('七天', dup_docs, 2)
rprint(f'   两条相同内容 -> 输出 {len(result)} 条')
rprint('   通过')

rprint('所有边界测试通过！')

边界场景测试:

1. 空列表测试:

输入 [] -> 输出 [] (应为 [])

通过

2. top_k 大于文档数:

...

通过

4. 重复内容测试:

两条相同内容 -> 输出 2 条

通过

所有边界测试通过！

## 6. 与 KnowledgeBase 集成测试（可选）

如果数据库已连接，可以运行此 cell 测试完整检索+重排流程。

In [ ]:
# from app.database import get_db
# from app.rag.retriever import get_kb_instance
# 
# async def test_full_pipeline():
#     async for db in get_db():
#         kb = get_kb_instance(db)
#         await kb.build_index()
#         
#         query = '七天无理由退换货'
#         rprint(f'查询: {query}')
#         
#         docs, detail = await kb.search_detailed(query, top_k=5)
#         
#         rprint(f'\n检索详情:')
#         for k, v in detail.items():
#             rprint(f'  {k}: {v}')
#         
#         rprint(f'\n最终结果:')
#         for i, c in enumerate(docs):
#             rprint(f'  {i+1}. [{c.title}] {c.content[:60]}...')
#         
#         return docs, detail
# 
# import asyncio
# docs, detail = await test_full_pipeline()